# LangGraph Agent Testing Notebook

This notebook focuses on end-to-end LangGraph behavior and helps pinpoint where failures occur.

## What this validates
- Graph runs end-to-end with valid output
- Fallback behavior when Neo4j is disabled
- Fallback behavior when Ollama times out/unavailable
- Explanation completeness and metadata diagnostics


In [ ]:
from pathlib import Path
import pandas as pd

from learning.debug_helpers import base_config, run_graph_with_timing, quick_output_checks, set_neo4j

## 1) Baseline graph run

In [ ]:
cfg = base_config('data/inventory_mock.csv')
payload, ms = run_graph_with_timing(cfg)
print(f'graph runtime: {ms:.2f} ms')
quick_output_checks(payload)

## 2) Neo4j-down fallback verification

In [ ]:
set_neo4j(False)
cfg = base_config('data/inventory_mock.csv')
payload_fallback, ms_fb = run_graph_with_timing(cfg)
print(f'graph runtime (neo4j disabled): {ms_fb:.2f} ms')
payload_fallback['metadata']['graph_source']

## 3) Ollama-timeout fallback verification
Set a dead localhost port and short timeout so template fallback should trigger.

In [ ]:
cfg_timeout = base_config('data/inventory_mock.csv')
cfg_timeout.setdefault('ollama', {})['base_url'] = 'http://127.0.0.1:65534'
cfg_timeout.setdefault('ollama', {})['timeout_ms'] = 10
payload_timeout, ms_to = run_graph_with_timing(cfg_timeout)
print(f'graph runtime (ollama timeout forced): {ms_to:.2f} ms')
checks = quick_output_checks(payload_timeout)
checks

## 4) Pinpoint issues from metadata

In [ ]:
errors = payload_timeout.get('metadata', {}).get('errors', [])
warnings = payload_timeout.get('metadata', {}).get('warnings', [])
print('errors:', len(errors))
print('warnings:', len(warnings))
pd.DataFrame(errors[:20]) if errors else 'No errors recorded'

In [ ]:
pd.DataFrame({'warning': warnings[:30]}) if warnings else 'No warnings recorded'

## 5) Output quality spot checks

In [ ]:
recs = payload_timeout.get('recommendations', [])
df = pd.DataFrame([{
    'sku_id': r.get('sku_id'),
    'status': r.get('status'),
    'context_source': r.get('context_source'),
    'confidence': r.get('confidence'),
    'explanation_len': len(str(r.get('plain_english_explanation', ''))),
} for r in recs])
df.head(10)

## Debug Checklist
Use this quick order when something breaks:
1. Run Notebook 1 MCP checks to isolate tool-level failures.
2. Check `metadata.errors` and `metadata.warnings` from graph output.
3. Confirm fallback source (`metadata.graph_source`, `context_source`).
4. Confirm every SKU has non-empty explanation.
5. Measure runtime and compare against expected budget.